# 📘 Notebook 18: Fine-Tuning a Model: End to End

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module E: Transformers & Fine-Tuning · Notebook 3 of 3 in this module · (18 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Walk through the complete **fine-tuning workflow** from data to evaluation
- Prepare and tokenise a custom dataset for a Transformer
- Fine-tune a pretrained model and **monitor training** for overfitting
- Evaluate the fine-tuned model with the metrics from Module C
- Apply **best practices** and understand the responsibilities that come with deploying a model

> **Prerequisites:** Notebooks 16-17, and conceptually the whole course.
>
> 🔭 **The finale.** Fine-tuning is where everything converges: data preparation (Module B), the train/test split and metrics (Module C), the training loop and overfitting vigilance (Module D), and pretrained Transformers (Module E). By the end you will have walked the full path from a single `print("Hello World")` to adapting a state-of-the-art model to your own problem.

> ℹ️ **Setup note.** Uses Hugging Face `transformers` and `datasets`. Full training is heavy for a browser kernel, treat the code as the exact recipe to run in Colab, a server, or a local GPU machine:
```python
import piplite
await piplite.install(['transformers', 'datasets', 'torch', 'scikit-learn'])
```

## 1. What fine-tuning actually does

### Intuition first
**Fine-tuning** takes a pretrained model, one that already understands language or images in general, and continues training it, gently, on **your** task-specific data. The model keeps its broad knowledge and specialises. Think of an experienced doctor doing a short focused course to specialise in one area: they do not relearn medicine, they adapt what they already know.

### Why a smaller learning rate
Because the model is already good, we nudge it with a **small learning rate** so we refine rather than destroy its existing knowledge. Too large a rate would wipe out the very pretraining we are trying to build on (a phenomenon called *catastrophic forgetting*). The learning-rate intuition from Notebook 09 returns one final time, with a new twist.

## 2. The fine-tuning workflow

Every fine-tuning project follows the same steps, and notice how each maps onto something you already know:

1. **Prepare the data**, collect labelled examples, clean them (Module B).
2. **Split** into train / validation / test (Module C, Notebook 08).
3. **Tokenise** the inputs with the model's tokeniser (Notebook 17).
4. **Load** the pretrained model with a fresh task head (Notebook 17).
5. **Train** with a small learning rate, watching the validation loss (Notebooks 09, 13).
6. **Evaluate** on the held-out test set with proper metrics (Notebook 10).
7. **Iterate** and, if good enough, deploy responsibly.

## 3. Preparing and tokenising data

We will fine-tune a small text classifier as a concrete example. The pattern is identical for any task. First, a tiny illustrative dataset (in practice you would have thousands of examples):

In [ ]:
# A small labelled dataset: 1 = positive, 0 = negative
texts = [
    "The lecture was clear and inspiring.",
    "I finally understand neural networks.",
    "This was confusing and poorly explained.",
    "A complete waste of my time.",
    "Brilliant examples that made it click.",
    "I could not follow any of it.",
]
labels = [1, 1, 0, 0, 1, 0]

# Train/test split -- the habit from Notebook 08
from sklearn.model_selection import train_test_split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.34, random_state=42)
print("Train size:", len(train_texts), " Test size:", len(test_texts))

In [ ]:
# Tokenise with the model's own tokeniser (Notebook 17)
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"   # a small, fast BERT variant
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_enc = tokenizer(train_texts, truncation=True, padding=True, return_tensors="pt")
test_enc  = tokenizer(test_texts,  truncation=True, padding=True, return_tensors="pt")
print("Tokenised input shape:", train_enc["input_ids"].shape)

## 4. Loading the model and fine-tuning

We load DistilBERT with a classification head sized for our 2 classes. The head starts random; the rest starts pretrained. Then we run the **same training loop** you have used since Notebook 13:

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

torch.manual_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# A SMALL learning rate -- we are refining, not retraining from scratch
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

train_labels_t = torch.tensor(train_labels)

model.train()
for epoch in range(4):
    optimizer.zero_grad()
    outputs = model(**train_enc, labels=train_labels_t)   # forward pass + loss
    loss = outputs.loss
    loss.backward()                                       # backprop (Notebook 13)
    optimizer.step()                                      # update (the downhill step)
    print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

The loss should decrease across epochs, the same signal of learning you first saw in Notebook 09. The four lines inside the loop (`zero_grad`, forward, `backward`, `step`) are exactly the universal loop. Only the model changed.

## 5. Monitoring training: watching for overfitting

### Why a validation set
Fine-tuning can **overfit** quickly, especially on small data, the very risk from Notebook 08. The defence is to watch performance on a held-out **validation set** during training. The tell-tale sign of overfitting: training loss keeps falling while validation loss starts **rising**. That divergence means *stop*, a practice called **early stopping**.

In [ ]:
import matplotlib.pyplot as plt

# Illustrative curves showing the classic overfitting signature
epochs = range(1, 11)
train_loss = [0.9, 0.7, 0.55, 0.42, 0.33, 0.27, 0.22, 0.18, 0.15, 0.13]
val_loss   = [0.92, 0.74, 0.6, 0.5, 0.45, 0.43, 0.44, 0.48, 0.55, 0.63]

plt.figure(figsize=(7, 4))
plt.plot(epochs, train_loss, "o-", label="training loss")
plt.plot(epochs, val_loss, "s-", label="validation loss")
plt.axvline(6, color="gray", linestyle="--", label="best epoch (stop here)")
plt.title("Training vs validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.show()

Read the plot: up to around epoch 6 both losses fall, the model is genuinely learning. After that, training loss keeps dropping but validation loss climbs, the model is now memorising training noise. The best model is the one at the **lowest validation loss**, not the lowest training loss.

## 6. Evaluating the fine-tuned model

Finally, we judge the model on the **test set** it never saw, using the classification metrics from Notebook 10: accuracy, and (for any imbalanced or high-stakes problem) precision, recall, and F1:

In [ ]:
import torch

model.eval()
with torch.no_grad():                          # no gradients needed for evaluation
    logits = model(**test_enc).logits
    preds = torch.argmax(logits, dim=1).tolist()

from sklearn.metrics import accuracy_score, classification_report
print("Test accuracy:", accuracy_score(test_labels, preds))
print(classification_report(test_labels, preds, zero_division=0))

## 7. Best practices and responsible use

### Practical best practices
- Start from the **smallest model** that could work; scale up only if needed.
- Use a **small learning rate** (often 1e-5 to 5e-5) to avoid catastrophic forgetting.
- Always keep a **validation set** and use **early stopping**.
- Set **random seeds** for reproducibility, a habit from Notebook 04 that matters most here.
- Keep a **held-out test set** untouched until the very end.

### Responsibility: especially in high-stakes domains
A fine-tuned model inherits whatever **biases** lived in its pretraining and your data. In sensitive applications, and medical AI, such as classifying brain MRI scans for signs of Alzheimer's disease, is a clear example, this is not optional diligence but an ethical requirement:
- Evaluate across **subgroups**, not just in aggregate; a model can be accurate overall yet fail a minority group.
- Prefer **high recall** when a missed positive is dangerous (recall the medical-screening discussion in Notebook 10).
- Be transparent about limitations; a model is a decision *aid*, not an oracle.
- Protect data **privacy** and obtain proper consent.

> 🧠 **The thread, complete.** The recall-versus-precision judgement that matters so much for a medical classifier is the very same idea introduced way back in Notebook 08 with the spam example. The mathematics scaled from a one-line formula to a fine-tuned Transformer, but the *reasoning* never changed. That continuity is the real lesson of this course.

---
## ✏️ Final Exercises

### Exercise 1
Looking at the training-vs-validation plot, suppose instead that **both** losses were still steadily decreasing at epoch 10. What would that tell you, and what would you do next?

In [ ]:
# Your answer here:


<details><summary>💡 Show solution</summary>

```python
# Both losses still falling means the model has NOT yet overfit and is still learning.
# The right move is to train for MORE epochs (and keep watching the validation loss),
# stopping when validation loss flattens or starts to rise.
```

*Early stopping is about reading these curves, not a fixed epoch count.*
</details>

### Exercise 2: *capstone reflection*
Outline, in your own words, the complete path you would take to fine-tune a model that classifies an image as one of three skin-condition categories, given ~500 labelled images. Name the key decision at each step and the notebook where you learned it.

In [ ]:
# Your outline here:


<details><summary>💡 Show solution</summary>

```python
# 1. Data: clean and inspect the 500 images (Module B).
# 2. Split into train/validation/test, stratified by class (Notebook 08).
# 3. Small data -> start with FEATURE EXTRACTION from a pretrained CNN (Notebooks 14, 17).
# 4. Replace the final layer for 3 classes; small learning rate (Notebooks 17, 18).
# 5. Train, watching validation loss; early-stop at its minimum (Notebooks 09, 13, 18).
# 6. Evaluate on the untouched test set; report per-class recall, not just accuracy,
#    and check subgroup performance (Notebooks 10, 18).
# 7. Document limitations and biases before any real-world use (Notebook 18).
```

*If you can write this outline confidently, you have internalised the whole course.*
</details>

## 🔑 Key takeaways

- **Fine-tuning** continues training a pretrained model on your data with a small learning rate, specialising without forgetting.
- The workflow reuses every earlier skill: data prep, splitting, tokenisation, the training loop, and proper metrics.
- **Monitor validation loss** and use **early stopping**: the lowest validation loss, not training loss, marks the best model.
- Evaluate on an untouched **test set**, attending to recall and subgroups in high-stakes settings.
- Powerful models carry **responsibilities**, bias, privacy, transparency, that are part of the engineering, not an afterthought.

### 🎓 Congratulations: you have completed the entire course!

You began by printing *Hello World* and you finish by fine-tuning a Transformer. Along the way you learned the Python language, the scientific stack, classical machine learning, the inner workings of neural networks, and the architecture behind modern AI, and you implemented the core of each idea **by hand** before letting a library take over.

That foundation is what distinguishes someone who truly understands these systems from someone who only calls their APIs. Wherever you take it next, research, industry, or teaching others, you now have the conceptual map to keep learning on your own.

Thank you for working through all eighteen notebooks.

---
*End of Module E: and the end of the course.*

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*